#Severity

In [0]:
-- 09_urgency: can the model flag the urgent (S1/S2) bugs?
-- Paste into the eval harness notebook. It reuses the prompts table and the
-- :model and :prompt_version parameters, but saves to its own table, because
-- the answer here is "urgent" / "not urgent", not a team.
 
-- ===== Cell U1 (run once): the urgency prompt =====
-- Based on Mozilla's severity definitions (S1 catastrophic, S2 serious = urgent; S3 normal, S4 minor = not).
INSERT INTO support.bugzilla.prompts
SELECT 'u1', 'You triage Firefox bug reports by urgency. Decide if a bug is URGENT or NOT URGENT.
 
URGENT means the bug is catastrophic or serious:
- crashes, hangs, or data loss (lost tabs, bookmarks, passwords, files, profile)
- a security hole or a privacy leak
- a core feature is broken for many users, such as pages failing to load, video failing to play, typing or clicking not working, or Firefox failing to start or update
- a recent regression that breaks a major website or a common workflow
- there is no reasonable workaround
 
NOT URGENT means the bug is normal or minor:
- it affects a less common feature, a specific site, or a rare setup
- there is a workaround
- it is cosmetic: layout glitches, wrong colors, text or wording problems
- it is a test failure, code cleanup, or a small enhancement
 
Most bugs are NOT URGENT. Only answer URGENT when the bug clearly matches the urgent list.'
WHERE 'u1' NOT IN (SELECT version FROM support.bugzilla.prompts);

-- ===== Cell U2 (run once): a table for urgency runs =====
CREATE TABLE IF NOT EXISTS support.bugzilla.urgency_runs (
  run_at TIMESTAMP, model STRING, prompt_version STRING,
  id BIGINT, team STRING, is_urgent BOOLEAN,
  predicted STRING, error STRING);

In [0]:
-- ===== Cell U3 (every run): set :model and :prompt_version (u1), then run =====
INSERT INTO support.bugzilla.urgency_runs
SELECT current_timestamp(), :model, :prompt_version,
       id, team, is_urgent, lower(trim(response.result)), response.errorMessage
FROM (
  SELECT e.id, e.team, e.is_urgent,
    ai_query(:model,
      CONCAT(p.team_guide,
             '\n\nBug title: ', e.summary,
             '\nBug description: ', substr(e.description, 1, 2000),
             '\n\nIs this bug URGENT or NOT URGENT? Answer with exactly one of: urgent, not urgent'),
      modelParameters => named_struct('max_tokens', 2000, 'temperature', 0.0),
      failOnError => false) AS response
  FROM support.bugzilla.urgency_set e
  JOIN support.bugzilla.prompts p ON p.version = :prompt_version
);

In [0]:
-- ===== Cell U4 (your scoreboard) =====
-- Per model and prompt_version, two numbers:
--   recall           = urgent bugs the model called 'urgent'  / all urgent bugs
--   false_alarm_rate = calm bugs the model called 'urgent'    / all non-urgent bugs
-- Hint: COUNT_IF(...) / COUNT_IF(...) works for both. Watch for answers that are neither word.
SELECT
  model,
  prompt_version,
  run_at,
  COUNT_IF(startswith(predicted, 'urgent') AND is_urgent) / COUNT_IF(is_urgent) AS recall,
  COUNT_IF(startswith(predicted, 'urgent') AND NOT is_urgent) / COUNT_IF(is_urgent = false) AS false_alarm_rate
FROM
  support.bugzilla.urgency_runs
GROUP BY
  model,
  prompt_version,
  run_at
ORDER BY
  model,
  run_at

In [0]:
SELECT * FROM support.bugzilla.urgency_runs

In [0]:
SELECT severity, team, summary, description FROM support.bugzilla.silver_bugs
WHERE id IN ("1906252", "1923208", "1976425", "1951930", "1928819", "1895865", "2003965", "1916843", "1958822", "1916508")

In [0]:
-- ===== Cell U1b (run once): add prompt u2 =====
INSERT INTO support.bugzilla.prompts
SELECT 'u2', team_guide || '
 
The bug has already been routed to a team, shown below. Some teams receive far more urgent bugs than others, so use the team as a hint, together with what the bug describes.
 
Answer with exactly one of these two phrases and nothing else: URGENT or NOT URGENT. Do not explain your answer.'
FROM support.bugzilla.prompts
WHERE version = 'u1'
  AND 'u2' NOT IN (SELECT version FROM support.bugzilla.prompts);

In [0]:
-- ===== Cell U3b (every u2 run): like U3, plus the team line =====
INSERT INTO support.bugzilla.urgency_runs
SELECT current_timestamp(), :model, :prompt_version,
       id, team, is_urgent, lower(trim(response.result)), response.errorMessage
FROM (
  SELECT e.id, e.team, e.is_urgent,
    ai_query(:model,
      CONCAT(p.team_guide,
             '\n\nTeam: ', e.team,
             '\nBug title: ', e.summary,
             '\nBug description: ', substr(e.description, 1, 2000),
             '\n\nAnswer with exactly one of: URGENT or NOT URGENT'),
      modelParameters => named_struct('max_tokens', 2000, 'temperature', 0.0),
      failOnError => false) AS response
  FROM support.bugzilla.urgency_set e
  JOIN support.bugzilla.prompts p ON p.version = :prompt_version
);

#Routing

In [0]:
-- 06_eval_harness: one notebook for every eval run.
-- Cell 1 runs ONCE (sets up two tables). After that you only touch Cell 2:
-- change the three parameters at the top of the notebook and press Run.

-- ===== Cell 1 (run once) =====
-- A table of prompt versions, so a prompt is saved once and picked by name.
CREATE TABLE IF NOT EXISTS support.bugzilla.prompts (version STRING, team_guide STRING);

INSERT INTO support.bugzilla.prompts
SELECT * FROM VALUES
('v1', 'You route Firefox bug reports to the team that should fix them. The teams are:
- Accessibility: screen readers, keyboard navigation, accessibility APIs.
- Accounts, Sync & Install: Firefox accounts, syncing between devices, installing and updating, importing from other browsers, enterprise policies, profile backup.
- Browser UI: the interface of the browser itself: tabs, address bar, search, bookmarks, history, downloads panel, sidebar, settings page, toolbars, themes, private browsing, session restore.
- Built-in Tools & AI: PDF viewer, screenshots, page translation, built-in AI and machine learning features.
- DOM & Web APIs: how web page code talks to the browser: HTML elements, forms, events, editing, copy and paste, drag and drop, navigation, workers, service workers, storage APIs such as IndexedDB and localStorage, notifications.
- Graphics: drawing pixels on screen: GPU rendering (WebRender), canvas, WebGL, WebGPU, images, color, smooth scrolling and zooming, VR.
- JavaScript: the JavaScript engine: JIT compiler, garbage collection, WebAssembly, the JavaScript standard library, debugger API.
- Layout & CSS: how pages are laid out and styled: CSS, flexbox, grid, text and font layout, tables, form control appearance, SVG, MathML, printing.
- Media & WebRTC: audio and video playback and recording, codecs, video calls (WebRTC), Web Audio, speech.
- Networking: HTTP, DNS, cookies, cache, proxies, WebSockets.
- New Tab & Messaging: the New Tab page, in-product messages and onboarding, experiments, remote settings.
- OS Integration: operating-system specific windows and widgets on Windows, macOS and Linux (GTK, Wayland), default browser setting, file handling, window management.
- Platform Internals: core plumbing: memory use, performance, the profiler, communication between processes (IPC), XPCOM, internationalization, spell checker, preferences, fuzzing.
- Privacy & Security: tracking protection, site permissions, certificates and TLS, sandboxing, saved passwords and logins, web authentication, cryptography.'),
('v2', 'You route Firefox bug reports to the team that should fix them. The teams are:
- Accessibility: the accessibility engine that assistive technology relies on: screen readers (NVDA, JAWS, Narrator, VoiceOver, Orca), ARIA roles and attributes, accessible names, focus and caret reporting to screen readers, high contrast and color contrast settings, accessibility tests (accname, aria WPT tests).
- Accounts, Sync & Install: Firefox accounts, syncing between devices, the installer and updater, importing from other browsers, enterprise policies, profile backup.
- Browser UI: the general interface of the browser: tabs and tab groups, address bar, search, bookmarks, history, downloads panel, sidebar, the settings page in general, toolbars, menus, themes, private browsing windows, session restore. Use Browser UI only when no more specific team below owns the feature.
- Built-in Tools & AI: PDF viewer, screenshots, page translation, built-in AI chatbots and machine learning features.
- DOM & Web APIs: how web page code talks to the browser: HTML elements, forms, events, editing, copy and paste, drag and drop, page navigation and back/forward history, window.open, iframes, workers, service workers, storage APIs such as IndexedDB, localStorage and the file system API, notifications, pointer lock, object URLs. Site compatibility bugs where a web page behaves wrongly usually belong here.
- Graphics: drawing pixels on screen: GPU rendering (WebRender), GPU driver problems (NVIDIA, AMD, Intel), canvas, WebGL, WebGPU, images, color, smooth scrolling and zooming, VR.
- JavaScript: the JavaScript engine: JIT compiler, garbage collection, WebAssembly, the JavaScript standard library, debugger API.
- Layout & CSS: how pages are laid out and styled: CSS, flexbox, grid, text and font layout, tables, form control appearance, SVG, MathML, printing.
- Media & WebRTC: audio and video playback and recording, codecs, video calls (WebRTC), Web Audio, speech.
- Networking: HTTP, DNS, cookies, the network cache, proxies, WebSockets, character encoding of downloaded pages.
- New Tab & Messaging: the New Tab page and its widgets (weather, timers, stories, shortcuts, the customize sidebar), in-product messages of every kind: promotions, infobars, feature callouts, onboarding screens, what is new pages, prompts asking the user to set Firefox as default or pin it, plus experiments (Nimbus) and remote settings.
- OS Integration: operating-system specific behavior on Windows, macOS and Linux (GTK, Wayland): native scrollbars, tooltips, cursors, fullscreen, window management, the taskbar and start menu, default browser setting, file and protocol handling.
- Platform Internals: core plumbing: memory use, performance, the profiler, communication between processes (IPC), XPCOM, internationalization, spell checker, preferences, fuzzing, build system.
- Privacy & Security: tracking protection (ETP) and its panel and exceptions, query parameter stripping, fingerprinting protection (RFP), site permissions and storage access, certificates, TLS and certificate error pages, sandboxing, saved passwords and the primary password, web authentication, cryptography, anti-malware and safe browsing.

Routing rule: pick the team that owns the feature or the code that is broken, not the place on screen where the problem shows up. For example, a certificate error page goes to Privacy & Security, an onboarding message goes to New Tab & Messaging, and a macOS scrollbar goes to OS Integration, even though all three appear in the browser interface.') AS t(version, team_guide)
WHERE version NOT IN (SELECT version FROM support.bugzilla.prompts);

-- One table that collects every run, one row per bug per run.
CREATE TABLE IF NOT EXISTS support.bugzilla.eval_runs (
  run_at TIMESTAMP, model STRING, prompt_version STRING, eval_table STRING,
  id BIGINT, true_team STRING, is_urgent BOOLEAN,
  predicted STRING, error STRING);

In [0]:
INSERT INTO support.bugzilla.prompts
SELECT 'v3', 'You route Firefox bug reports to the team that should fix them. Each team owns a set of Mozilla Bugzilla components, listed below with Mozilla''s own description of each component.

TEAM: Accessibility
- Disability Access: Accessibility-related bugs in Firefox: screen readers, support for magnifier tools, user interface bugs affecting the experience of people with disabilities, compliance with national and international accessibility standards.
- Disability Access APIs: Description: This component relates to bugs in our support for accessibility APIs on the various platforms. Accessibility APIs allow 3rd party products, such as screen readers used by visually impaired users, to communicate with our content and UI. The APIs we support specifically are MSAA on Windows and ATK on UNIX/Linux (Apple has not yet published specs for an accessibility API on OS X). This component is not for keyboard, focus or any accessibility bugs other than those relating to the APIs we export.
- Keyboard Navigation: Keyboard shortcut navigation in Firefox''s user interface

TEAM: Accounts, Sync & Install
- AutoConfig (Mission Control Desktop): For bugs using an AutoConfig (mozilla.cfg) file to configure Firefox or other Mozilla applications. NOT for Thunderbird Autoconfiguration.
- Enterprise Policies: Bugs and feature requests related to specific policies implemented in Firefox, or to the policy engine itself. Bugs about Group Policy Object (GPO) should be reported in this component.
- Firefox Accounts: For bugs in shared services/fxaccounts code related to Firefox accounts
- Installer: Bugs and feature requests for the Firefox application install wizard.
- Migration: Firefox Profile Migration from other browsers.
- Profile Backup: Bugs and feature requests for profile backup in Firefox
- Sync: Bugs and feature requests for Sync in Firefox Desktop.

TEAM: Browser UI
- Address Bar: For bugs in the Firefox "Smart Location Bar" (aka Awesome Bar) UI element, including the bookmark and history autocomplete matching behavior.
- Bookmarks & History: Bugs and feature requests for Firefox bookmarks & history. This includes the bookmarks menu, add and modify bookmarks dialogs and the bookmarks manager.
- Distributions: Bugs related to the creation of desktop distributions (distribution.ini)
- Downloads Panel: For issues with the Downloads Panel, the new download manager of Firefox that lives in the navigation toolbar.
- Firefox View: Bugs relating to the Firefox View page, toolbar button and integration
- Headless: For bugs related to headless browsing mode where Firefox is a web browser without a Graphical User Interface (GUI).
- IP Protection: Bugs related to ''Firefox VPN''
- Menus: Bugs and feature requests for Firefox menus. This includes toplevel and context menus.
- Normandy Client: In-browser component that validates and executes actions in a sandbox
- Page Info Window: For issues with the Page Info window.
- Pocket: Bugs involving the integration with Pocket and Firefox.
- Private Browsing: For bugs and feature requests in Firefox''s Private Browsing implementation.
- Search: Web search UI, default search engines and Toolkit''s search service. Find-in-page bugs belong in Toolkit::Find Toolbar instead.
- Session Restore: For bugs in Firefox''s session restore functionality, including the undo close tab feature.
- Session Restore: Session Restore is the platform independent component for collecting and restoring tab and document state.
- Settings UI: Bugs and feature requests for Firefox Settings. This is specific to the Tools->Settings UI - for issues with other types of settings (e.g. permissions, settings internal to websites, OS settings, ...), requests to change default settings values, bugs where internal values of settings have no effect, etc., file bugs in components related to those features/settings instead.
- Sidebar: Bugs and feature requests for Firefox Sidebar
- Site Identity: For bugs and feature requests in the UI that signals site identity and connection security, such as the lock icon and the corresponding “identity panel”.
- Tabbed Browser: For problems in the browser tab features or problems with the widget itself.
- Tabbed Browser: Split View: For bugs or feature work relating to splitview-specific tab and content behavior.
- Tabbed Browser: Tab Groups: Bugs and feature requests related to tab groups in the tab strip and other places that tabs live.
- Theme: General user interface, user experience, and visual design for the default theme used in Firefox.
- Toolbars and Customization: Bugs and feature requests for Firefox toolbars. This includes toolbar customization.
- Tours: UITour, firstrun and whatsnew Firefox bugs.
- Web Apps: This component is responsible for web apps for Firefox.
- WebPayments UI: User Interface for the WebPayments <a href="https://w3c.github.io/browser-payment-api/">Payment Request API</a> and <a href="https://w3c.github.io/payment-handler/">Payment Handler API</a>.

TEAM: Built-in Tools & AI
- Machine Learning: Frontend: Client UIs, AI components, browser AI integrations
- Machine Learning: General: Models, security, general triage, etc. for AI features
- Machine Learning: On Device: Local inference, platform integrations, AI data pipelines
- PDF Viewer: For bugs related to Firefox''s built-in PDF viewing capabilities (also known as pdf.js).
- Screenshots: Bugs and feature requests for Firefox Screenshots
- Translations: Instant translation and language detection in Firefox.

TEAM: DOM & Web APIs
- DOM: Animation: This component covers the <a href="https://w3c.github.io/web-animations/">Web Animations</a> implementation in Gecko.
- DOM: Bindings (WebIDL): Issues with code implementing WebIDL, <strong>not</strong> WebIDL itself
- DOM: CSS Object Model: <p>Issues with CSSOM support and other CSS-related DOM APIs. This is not the place for most CSS bugs unless they involve scripting; see the CSS and Layout components instead.</p> <p>Example bugs:</p> <ul> <li>document.styleSheets doesn’t include style sheets from file:// URLs</li> <li>crash setting element.style.color to "red"</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Font Loading</li> <li>CSSOM</li> <li>CSSOM View Module</li> <li>Geometry Interfaces</li> </ul>
- DOM: Content Processes: Issues related to implementation of content processes in Firefox, especially the code in dom/ipc ([more info](https://wiki.mozilla.org/Project_Fission))
- DOM: Copy & Paste and Drag & Drop: All issues with copy-and-paste and drag-and-drop can be reported here, be it with the web API or the platform backends. This includes clipboard issues.
- DOM: Core & HTML: Issues in DOM tree <a href="https://dom.spec.whatwg.org/">https://dom.spec.whatwg.org</a> & HTML <a href="https://html.spec.whatwg.org/">https://html.spec.whatwg.org</a> that do not fit into any other DOM or HTML component or which span multiple DOM or HTML components.
- DOM: Credential Management: The implementation of the <a href="https://w3c.github.io/webappsec-credential-management">Credential Management API</a> and other APIs that define new <a href="https://w3c.github.io/webappsec-credential-management/#the-credential-interface">Credentials</a> but do not have their own component.
- DOM: Device Interfaces: This component covers APIs exposing device interfaces for communication, networking, multimedia, etc.
- DOM: Editor: For bugs with editing text on web pages. Related HTML features are &lt;input type=text>, &lt;textarea>, contenteditable, and the designMode API.
- DOM: Events: This component includes bugs for DOM events from <a href="https://dom.spec.whatwg.org/">https://dom.spec.whatwg.org</a>.
- DOM: File: For bugs in the DOM Blob, DOM File and FileSystem APIs
- DOM: Forms: For bugs with <a href="https://html.spec.whatwg.org/multipage/forms.html">HTML forms</a>. (See also Layout: Form Controls.)
- DOM: Geolocation: The geolocation API and framework enables reporting a physical location, in cooperating with a location provider (such as a GPS).
- DOM: HTML Parser: This system consumes content from the web, parses, validates and builds a content model (document)
- DOM: Navigation: For bugs with HTML’s <a href="https://html.spec.whatwg.org/multipage/browsing-the-web.html#navigate">navigate</a> algorithm and browsing contexts.
- DOM: Networking: Networking-related DOM APIs, such as XmlHttpRequest, Fetch, Web Sockets, EventSource, and Beacon.
- DOM: Notifications: The component that covers Web Notifications API and the integration with the corresponding OS native notification APIs.
- DOM: Performance APIs: Bugs affecting the implementation of the DOM performance APIs such as ElementTiming.
- DOM: Push Subscriptions: Subscribing push using PushManager and waking up the corresponding service worker when a remote server sends a push message.
- DOM: Selection: This component includes bugs for selection handling from <a href="https://w3c.github.io/selection-api/">https://w3c.github.io/selection-api/</a>.
- DOM: Serializers: Bugs involving conversion to html or plaintext, either during saving or during copy/paste. This component is NOT for general copy/paste bugs, for bugs involving save dialogs or writing to files when saving, or for I18n-related bugs; only for problems with the actual text which is saved.
- DOM: Service Workers: This component covers the <a href="https://github.com/slightlyoff/ServiceWorker/">Service Worker</a> implementation in Gecko.
- DOM: Streams: Implementation of WHATWG Streams specification
- DOM: UI Events & Focus Handling: This component includes bugs for user input events, for example <a href="https://w3c.github.io/uievents">https://w3c.github.io/uievents</a>, focus handling from <a href="https://html.spec.whatwg.org">https://html.spec.whatwg.org</a>, and access key handling on a page.
- DOM: Web Payments: This component covers the WebPayments <a href="https://w3c.github.io/browser-payment-api/">Payment Request API</a> and <a href="https://w3c.github.io/payment-handler/">Payment Handler API</a> in Gecko.
- DOM: Web Share: This component covers the <a href="https://w3c.github.io/web-share"> Web Share API]</a> and <a href="https://w3c.github.io/web-share-target">Web Share Target API</a> in Gecko.
- DOM: Window and Location: For issues with the Window global object, Location object, and the WindowProxy object.
- DOM: Workers: This component covers the <a href="https://developer.mozilla.org/En/DOM/Worker">Web Workers</a> implementation in Gecko.
- DOM: postMessage: For bugs with all postMessage() APIs (i.e., with serializing, deserializing, and transferring), including <a href="https://html.spec.whatwg.org/multipage/web-messaging.html#broadcasting-to-other-browsing-contexts">BroadcastChannel</a>, <a href="https://html.spec.whatwg.org/multipage/web-messaging.html#message-channels">MessageChannel</a>, and <a href="https://html.spec.whatwg.org/multipage/web-messaging.html#web-messaging">window.postMessage()</a>.
- Storage: Bucket File System: Bucket File System implementation, exposed through self.navigator.storage.getDirectory().
- Storage: Cache API: Request/Response storage using the service worker’s Cache API. I.e., CacheStorage exposed through self.caches.
- Storage: IndexedDB: <a href="https://w3c.github.io/IndexedDB/">IndexedDB</a> implementation, exposed through self.indexedDB.
- Storage: Quota Manager: For bugs in Gecko''s centralized management of storage and disk usage quota in user profiles.
- Storage: StorageManager: StorageManager from the <a href="https://storage.spec.whatwg.org/">Storage standard</a>, exposed through navigator.storage.
- Storage: localStorage & sessionStorage: For bugs with <a href="https://html.spec.whatwg.org/multipage/webstorage.html#the-localstorage-attribute">localStorage</a> and <a href="https://html.spec.whatwg.org/multipage/webstorage.html#the-sessionstorage-attribute">sessionStorage</a>.
- XML: For bugs in our XML handling. For example, parser errors, or errors in wellformedness error reporting.
- XSLT: For bugs on Mozilla''s XSLT support. See the <a href="http://www.mozilla.org/projects/xslt">project page</a> for tips on testcases and commonly encountered legacy problems.

TEAM: Graphics
- Graphics: Mapping of cross platform rendering interfaces to various 2D graphics APIs
- Graphics: Canvas2D: Bugs with the HTML5 <a href="http://whatwg.org/specs/web-apps/current-work/#dynamic">&lt;canvas&gt;</a> element, its XUL sibling, and related rendering contexts.
- Graphics: CanvasWebGL: Bugs with the WebGL Canvas Context.
- Graphics: Color Management: qcms and other issues relating to Mozilla''s color management implementation.
- Graphics: Image Blocking: For bugs about (un)blocking images from servers.
- Graphics: ImageLib: ImageLib decodes GIF, JPEG and PNG images, and provides the decoded data to the Compositor for display. If Firefox or Seamonkey can display an image when loaded separately from the page, ImageLib is working, and the actual imaging bug exists elsewhere within Firefox or Seamonkey.<br> <br> Examples of appropriate bugs: PNG gAMA chunk ignored; Crashes on GIF w/corrupted frame(merr-01.gif); or PNGs and JPEGs aren''t displayed on FreeBSD.
- Graphics: Text: <p>Issues with glyph rasterization and painting, font format support, and interfacing with platform font APIs. For issues with font selection, text shaping, and glyph selection, see the Layout: Text and Fonts component.</p>
- Graphics: WebGPU: Issues related to implementation of WebGPU API in Gecko. This category includes: validation/tracking/security issues, graphics abstraction issues on various platforms, integration with WebRender for presenting, and WebIDL bindings.
- Graphics: WebRender: Bugs related to the integration of WebRender into Gecko.
- Panning and Zooming: Covers issues relating to the asynchronous panning and zooming (APZ) module, including scrolling and gesture detection. Issues with reflowing ("full") zoom should be filed in Core :: Layout (for issues with page rendering when zoomed) or Firefox :: Toolbars and Customization (for issues with the zoom level indicator UI) instead.
- Web Painting: Construction of display lists, hit-testing, building of Layers and painting.
- WebVR: Bugs with the WebVR functionality, including devices, graphics and integration

TEAM: JavaScript
- JavaScript Engine: The interpreter engine for the core JavaScript language, independent of the browser''s object model. File ONLY core JavaScript language bugs in this category. For bugs involving browser objects such as "window" and "document", use the "DOM" component. For bugs involving calls between JavaScript and C++, use the "XPConnect" component.
- JavaScript Engine: JIT: JavaScript engine''s JIT compilers
- JavaScript: Debugger API: Implementation of the Debugger API in SpiderMonkey.
- JavaScript: GC: File bugs here for issues involving the garbage collector (crashes, leaks, etc.), where the issue is <em>not</em> one caused by failure to use the JSAPI correctly.
- JavaScript: Internationalization API: File bugs here for issues related to anything exposed through the Intl property on the global object.
- JavaScript: Standard Library: File bugs here for issues related to non-conforming behaviors in any of JavaScript''s standard class constructors, methods exposed on instances through the prototype chain, and so on. (Examples: String.prototype.charAt doesn''t work on non-String objects, the Object constructor doesn''t work correctly when passed <code>document.all</code>, etc.)<br> <br> If the relevant functionality has a more-precise component, bugs within it should be filed there (e.g. Intl.NumberFormat bugs should be filed in the JavaScript: Internationalization API component).
- JavaScript: WebAssembly: Implementation of the WebAssembly (wasm) standard, including compilation and runtime support.
- XPConnect: Facilitates calling between JavaScript and XPCOM components.
- js-ctypes: js-ctypes is a ffi (Foreign Function Interface) component for JS/XPCOM. js-ctypes allows developers to call functions in dlls/shared libraries directly from JavaScript without the need to create binary XPCOM wrappers.

TEAM: Layout & CSS
- CSS Parsing and Computation: <p>Issues with loading CSS style sheets from the network, parsing style sheets and style attributes in HTML markup, performing the CSS cascade, selector matching, and producing correct computed values for CSS properties.</p> <p>Example bugs:</p> <ul> <li>toggling class name on an element does not restyle the element</li> <li>background-color property does not support hsl() values</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Cascading</li> <li>CSS Conditional</li> <li>CSS Logical Properties</li> <li>CSS Namespaces</li> <li>CSS Scoping</li> <li>CSS Style Attributes</li> <li>CSS Syntax</li> <li>CSS Values</li> <li>CSS Variables</li> <li>Media Queries</li> <li>Selectors</li> </ul>
- CSS Transitions and Animations: <p>Issues with animating and transitioning CSS properties. For issues with the Web Animations API, see the DOM: Animations component.</p> <p>Example bugs:</p> <ul> <li>transitions are not triggered when the display property is changed simultaneously</li> <li>the font-weight property animates discretely but should be interpolated smoothly</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Animations</li> <li>CSS Timing</li> <li>CSS Transitions</li> <li>Motion Path</li> </ul>
- Layout: <p>Issues in Layout that do not fit into any other Layout component or which span multiple Layout components.</p> <p>Bugs related to the top level presentation objects (pres shell, pres context, and document viewer), the frame constructor, and the base frame classes, as well as general issues with alignment and sizing, all belong here.</p>
- Layout: Block and Inline: <p>Issues with core layout and geometry for basic block and inline elements. This includes issues with ::first-letter and ::first-line.</p> <p>Example bugs:</p> <ul> <li>margins between two blocks do not collapse properly</li> <li>button does not align its baseline correctly with the rest of the line it’s in</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Box</li> <li>CSS Inline Layout</li> </ul>
- Layout: Columns: <p>Issues with CSS multi-column layout.</p> <p>Example bugs:</p> <ul> <li>column rules are not properly drawn on overflow columns</li> <li>column gap should be included in minimum width calculation when the number of columns is non-auto</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Multicol</li> </ul>
- Layout: Flexbox: <p>Issues with CSS flexible box layout.</p> <p>Example bugs:</p> <ul> <li>flex items not wrapping in multi-line flexbox with "overflow:hidden"</li> <li>baseline for empty flexbox container seems wrong</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Flexbox</li> </ul>
- Layout: Floats: <p>Issues with CSS floats and clearance.</p> <p>Example bugs:</p> <ul> <li>float moves to wrong position when you dynamically set "clear" property on ancestor</li> <li>inline frames without enough inline space should clear floats, regardless of shape-outside</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS 2.2 (§ Floats, and others)</li> <li>CSS Shapes</li> </ul>
- Layout: Form Controls: <p>Issues with the layout and appearance of HTML form controls, such as buttons, date/time pickers, select elements, and textareas.</p> <p>Example bugs:</p> <ul> <li>datepicker remains visible after tabbing out of date input field</li> <li>the filename field of &lt;input type=file&gt; renders incorrectly in vertical writing mode with dir=rtl</li> </ul> <p>Specifications covered:</p> <ul> <li>HTML (§ Forms)</li> </ul>
- Layout: Generated Content, Lists, and Counters: <p>Issues with handling of CSS generated content, list items, and counters.</p> <p>Example bugs:</p> <ul> <li>::before and ::after are not rendered correctly on &lt;details&gt;</li> <li>list item bullet characters don''t display in the specified font face</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Counter Styles</li> <li>CSS Generated Content</li> <li>CSS Lists</li> <li>CSS Pseudo-Elements</li> </ul>
- Layout: Grid: <p>Issues with CSS grid layout.</p> <p>Example bugs:</p> <ul> <li>stretching the inline size of &lt;button&gt; grid items doesn''t work</li> <li>abspos child of a grid should respect "align-self"/"justify-self" when computing its static position</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Grid Layout</li> </ul>
- Layout: Images, Video, and HTML Frames: <p>Issues with the CSS images and other replaced content. This includes loading, layout, and rendering of CSS images (such as in background-image), and the layout of HTML img, picture, video, iframe, frame, and frameset elements. For issues with image decoding and video playback, see the ImageLib and Audio/Video components respectively.</p> <p>Example bugs:</p> <ul> <li>SVG as image with an intrinsic width and max-height has its width incorrectly scaled down by the same ratio as the height is scaled down</li> <li>&lt;iframe&gt; has incorrect default size in vertical writing mode</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Backgrounds</li> <li>CSS Images</li> </ul>
- Layout: Positioned: <p>Issues with relative, absolute, fixed, and sticky positioning.</p> <p>Example bugs:</p> <ul> <li>&lt;thead&gt; with position: sticky gets stuck after scrolling down</li> <li>wrong absolute positioning when the containing block has a min-height and overflow:hidden</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Positioned Layout</li> </ul>
- Layout: Ruby: <p>Issues with CSS Ruby text layout.</p> <p>Example bugs:</p> <ul> <li>ruby annotation is not placed correctly for RTL content</li> <li>ruby base after a line break may disappear</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Ruby</li> </ul>
- Layout: Scrolling and Overflow: <p>Issues with the layout and rendering of scrollable elements, scrollbars, and with the handling of overflowing content. Also includes issues with rendering of the regular text caret and the accessible caret. For problems with rendering of currently scrolling elements, see the Panning and Zooming component.</p> <p>Example bugs:</p> <ul> <li>&lt;legend&gt; w/ display:flex overflow:scroll sometimes renders without scrollbars</li> <li>overflow: hidden on fieldsets spills over</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Scroll Anchoring</li> <li>CSS Scroll Snap</li> <li>CSS Scrollbars</li> <li>CSS Overflow</li> </ul>
- Layout: Tables: <p>Issues with table layout and rendering.</p> <p>Example bugs:</p> <ul> <li>border-style: groove not applied for tables with border-collapse: collapse</li> <li>incorrect width for table cells with padding, box-sizing: border-box, and table-layout: fixed</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS 2.2 (§ Tables)</li> </ul>
- Layout: Text and Fonts: <p>Issues with CSS font selection, font loading, text shaping and glyph selection, breaking, text decorations, bidirectional text processing, and white space processing. For issues with glyph rendering and font format support, see the Graphics: Text component.</p> <p>Example bugs:</p> <ul> <li>line-breaking code does not handle supplementary-plane characters properly</li> <li>double or wavy decoration lines are not repainted correctly if specified on floating first letter</li> </ul> <p>Specifications covered:</p> <ul> <li>CSS Fonts</li> <li>CSS Size Adjustment</li> <li>CSS Text</li> <li>CSS Text Decoration</li> </ul>
- MathML: Mathematical Markup Language
- Print Preview: <p>Issues with layout and rendering a document in Print Preview mode. For problems in printed or PDF output that do not appear in Print Preview, see the Printing: Output component.</p<>
- Printing: Output: Problems printing from this product should be submitted under this component.
- Printing: Setup: Integration relating to the OS printing system, including cross-platform components such as the page setup dialog.
- SVG: <p>Issues with the layout or rendering of SVG content and interacting with the SVG DOM.</p> <p>Example bugs:</p> <ul> <li>SVG marker can not be clipped with clip-path</li> <li>x position adjustments of SVG vertical text on a path moves in the wrong direction</li> </ul> <p>Specifications covered:</p> <ul> <li>SVG</li> </ul>
- XUL: <p>For bugs in the <A HREF="http://www.mozilla.org/xpfe/xptoolkit/xulintro.html">XUL language</A>. (XUL is an XML-based language which specifies Mozilla''s user interface.)</p> <p>If you''re not actually writing XUL code, this is probably not the right place for your bug.</p> <p>Examples of appropriate bugs: Duplicate window ID causes infinite loop; or Scrollbars disappear after loading XUL document</p>

TEAM: Media & WebRTC
- Audio/Video: For problems related to media (video/audio) -- especially when the reporter is not sure which area of the media stack the problem is in. This category will mostly be for untriaged video/audio issues. Bugs in this category will typically be moved to another media category during the triage process.
- Audio/Video: GMP: Any bug in the Gecko Media Plugin framework. This is the code that glues the openh264 plugin and EME plugins to the browser.
- Audio/Video: MediaStreamGraph: Any bug related to MediaStreams, MediaStreamTracks, TrackUnions and the MediaStream GraphDriver.
- Audio/Video: Playback: For problems related to the HTML 5 media elements (&lt;video&gt; and &lt;audio&gt;) -- including WebM, MP4, MSE and EME issues. This would also typically include decoding problems in the codecs themselves (e.g. VP8, VP9, H.264, AAC) experienced during playback.
- Audio/Video: Recording: Issues relating to recording/encoding of audio and video using the HTML5 MediaRecorder API.
- Audio/Video: Web Codecs: Bugs and enhancement requests relating to the Web Codecs API
- Audio/Video: cubeb: Any bug in libcubeb. Typically audio output and input bugs would go here
- Web Audio: Bugs and enhancement requests relating to the Web Audio support.
- Web Speech: This component covers the <a href="https://dvcs.w3.org/hg/speech-api/raw-file/tip/webspeechapi.html">Web Speech API</a> implementation in Gecko.
- WebRTC: General bugs associated with WebRTC
- WebRTC: Audio/Video: Bugs in WebRTC audio and video capture, handling, echo cancellation, encoding, and playback
- WebRTC: Networking: Bugs in the networking portions of WebRTC (PeerConnection dataChannels, SCTP, DTLS, SRTP, ICE, TURN, STUN, etc)
- WebRTC: Signaling: Bugs in WebRTC call-setup and re-negotiation code, in particular JSEP and SDP handling

TEAM: Networking
- Networking: For bugs in Mozilla''s modular networking library (aka "Netlib" or "Necko".) The networking library supplies the software interface that Mozilla uses to access physical transports (e.g. the Internet and local drives), perform URL resolutions, and handle a variety of networking protocols. <P>Examples of appropriate bugs: URLs with backslash not fetched; URLs starting with a single slash turn into http:///; Cannot access authenticated FTP site.
- Networking: Cache: For bugs in the cache, which stores recently accessed data for more rapid access. Includes the disk cache, memory cache, and all cache preferences.<BR><BR> Examples of appropriate bugs: Implement compression of network cache data; or Implement partial HTTP caching.
- Networking: Cookies: A general mechanism which server side connections (such as CGI scripts) can use to both store and retrieve information on the client side of the connection. This refers to HTML cookies; little blobs of data we store and share with sites
- Networking: DNS: For bugs in Mozilla''s implementation of DNS (Domain Name Service)
- Networking: File: For bugs in Mozilla''s File IO code. This component deals with File specific issues like writing to disk, reading local files, etc Related problems with file:// protocol may also be addressed here.
- Networking: HTTP: For bugs in Mozilla''s HTTP networking code. This component deals with HTTP specific issues like pipelining, keep-alive, HTTP proxies, 1.1 issues, redirects, authentication (basic), etc. <p> Examples of appropriate bugs: Problems with proxy authentication; HTTP redirects looping indefinitely, etc.
- Networking: JAR: For bugs relating to the jar protocol handler, .jar file handling, and .jar directory browsing.
- Networking: Proxy: For bugs in Gecko''s proxy code. This component covers issues relating to proxy configuration, HTTP, HTTPS, SOCKS v4/v5 proxy support.
- Networking: WebSockets: Bugs related to the performance/correctness of the on-the-wire WebSocket networking protocol: file other WebSockets bugs under DOM:WebSockets interface may be better classified as Core::DOM.

TEAM: New Tab & Messaging
- Messaging System: For bugs that involve user interrupting messages, badging and notifications in Firefox. This includes Contextual Feature Recommender (CFR), Onboarding Tour, Feature Callouts, Menu/Toolbar badges and door-hangers.
- New Tab Page: The page displayed when a new tab is opened (about:newtab).
- Nimbus Desktop Client: All bugs related to Nimbus experimentation client code in Firefox Desktop
- Remote Settings Client: Module in charge of fetching and keeping remote settings in sync with Mozilla servers.

TEAM: OS Integration
- DLL Services: Bugs relating to Windows dynamic linker instrumentation, the DLL interceptor, or DLL blocking
- File Handling: For issues dealing with helper applications, and guessing Content Types when they aren''t specified/known (ftp:, file:, jar:, but generally not http:). This component does not cover: backend networking issues, such as those covered by Networking: FTP or Networking: File, nor does it cover the Download Manager which has its own component in the Toolkit product.
- Shell Integration: This component is responsible for determining if Firefox is the default browser, setting Firefox as the default browser, and setting the desktop background.
- Widget: Cross platform widget interfaces and cross platform base implementations. Bugs that belong in this component are cross-platform problems and <a href="http://lxr.mozilla.org/seamonkey/find?string=widget/public/.*idl" >API</a> issues. Bugs like this that only occur on one platform belong in the appropriate component listed below.
- Widget: Cocoa: Mapping of cross platform widget interfaces to Mac Cocoa APIs.
- Widget: Gtk: Mapping of cross platform widget interfaces to GTK platform APIs (both versions). Wayland bugs belong in this component.
- Widget: Win32: Mapping of cross platform widget interfaces to Win32 platform APIs.
- Window Management: Creation and management of top-level windows

TEAM: Platform Internals
- Cycle Collector: The cycle collector automates the breaking of cycles involving reference counted C++ objects and GC-managed JS objects.
- Find Backend: For bugs about the "Find in page" feature.
- Fuzzing: For bugs related to fuzzing code or team requests that don''t fit anywhere else. Changes to individual fuzzing targets should be filed in the respective components.
- Gecko Profiler: The built-in profiler which can be used in order to get profiling information from within Gecko.
- Hardware Abstraction Layer (HAL): For bugs in the Hardware Abstraction Layer (HAL) code regardless of the backend. Example of appropriate bugs: implementation of a new feature for a backend, fixing a bug for a specific backend or general bugs regarding HAL.
- IPC: Core inter-process communication and IPDL system for communicating with and isolating sub-processes. Please file plugin-specific IPC bugs in Core: Plug-Ins and content-specific IPC bugs in Core: DOM: Content Processes.
- Internationalization: Internationalization is the process of designing and developing a software product to function in multiple locales. This process involves identifying the locales that must be supported, designing features which support those locales, and writing the code needed.
- Internationalization: Localization: Localization is the process of adapting software for a specific international market; this process includes translating the user interface, resizing dialog boxes, replacing icons and other culturally sensitive graphics (if necessary) as well as customizing features.
- MFBT: File bugs here for headers, macros, data structures, methods, and other functionality which should be available to SpiderMonkey and to all Gecko code, implemented in the mfbt/ source directory.
- Memory Allocator: Most things related to memory allocation in Gecko, including jemalloc, replace-malloc, logalloc, etc. For reports about memory usage, please use the Firefox::Untriaged or Thunderbird::Untriaged component instead.
- Performance: Issues related to the performance of the browser front-end.
- Performance Engineering: A planned task for the Performance Engineering team which has been reviewed, and internally discussed as work to be scheduled and completed. Not to be used for general Performance issues.
- Performance: General: Issues related to Performance that we cannot yet categorize clearly in any of the Performance sub-components.
- Performance: Memory: Issues related to excessive browser memory usage, including when the memory usage is responsible for direct performance degradation.
- Performance: Navigation: Issues related to both hard and soft navigations on the web.
- Performance: Responsiveness: Issues related to responsiveness when interacting with content on websites without navigating to different pages. (For example zooming or panning Google Maps)
- Preferences: Backend: For bugs in libPref, the backend library that reads, parses, and writes preferences to the user''s hard disk. Unless you''re a Mozilla developer, your bug probably doesn''t belong in this component. <p>Examples of appropriate bugs: prefs.js on the mac has non-mac style line endings; Large libpref string causes dialogs to crash on Linux; or JS Error at startup reading prefs file on migrated profile.
- SQLite and Embedded Database Bindings: SQLite-backed generic data storage and query engine.
- Sanitizers: For issues related to sanitizers - e.g. ASan, LSan, TSan, UBSan, . Feature requests for sanitizers can also be filed here.
- Spelling Checker: en-US Dictionary: For missing or misspelled words in the en-US dictionary shipping as part of Mozilla desktop applications.
- Spelling checker: For bugs with the spelling checker used by Mozilla-based applications. This includes the backend code, the Hunspell spell checker, and the osxspell interface to the macOS spell checker.
- String: The hierarchy of classes related to managing character sequences.
- XPCOM: This is the basis of our component technology; this covers the mozilla/xpcom source directory and includes the "repository". Unlikely a tester would be able to tell there was an XPCOM problem specifically.
- mozglue: For issues in the Android glue, the custom dynamic linker or other parts of the mozglue library.

TEAM: Privacy & Security
- DOM: Security: Native content-based security features including: Content Security Policy (CSP), Mixed Content Blocker (MCB), and Safe Browsing.
- DOM: Web Authentication: Bugs and issues related to the W3C Web Authentication (webauthn) implementation in Firefox, or its interaction with operating system and hardware components.
- DOM: Web Crypto: Bugs and issues related to the W3C Web Crypto implementation in Firefox.
- Data Loss Prevention: For bugs in Firefox''s support of DLP (Data Loss Prevention) products and the Content Analysis SDK including DLP-related prompts, warnings, and performance.
- Permission Manager: For bugs related to the permission manager but not the permission management. Example of appropriate bugs: permissions in the database are ignored or removing permission doesn''t work. Example of inappropriate bugs: feature F should use permission P or any user interface related bugs.
- Privacy: Anti-Tracking: Tracking detection and content-blocking; part of the Enhanced Tracking Protection functionality
- Protections UI: For bugs and feature requests in Firefox’s Anti-Tracking Protections UI such as the Protection Panel and the Protection Report.
- Security: For app-level security bugs. <p> If the problem relates to underlying components (PSM, NSS, Core, Toolkit) then please file it in the appropriate product instead of here.
- Security: For non-crypto security bugs. Or use this general component if you know you have a Browser security bug, but not sure what component to use. Bug will be reviewed for proper component.
- Security: CAPS: Caps is the capabilities-based security system for the browser. Manages principals (entities responsible for a piece of code) and the protected capabilities (the ability to perform a restricted action) that may be enabled for given principals.
- Security: PSM: The TLS configuration layer between Gecko and NSS. Handles certificate discovery, verification, and revocation checking.
- Security: Process Sandboxing: For process sandboxing bugs. Generally this covers code under the security/sandbox/ source directory, but there is code elsewhere to hook the different platform process sandboxes into Gecko etc.
- Security: RLBox: RLBox / WASM Sandboxing
- Site Permissions: Bugs and feature requests for the Firefox UI to give or revoke web page permissions to access device functionality such as camera, microphone, geolocation or notifications.
- about:logins: Issues with the <a href="https://wiki.mozilla.org/Toolkit:Password_Manager/about:logins" target="_blank" rel="noopener">Firefox Lockwise ''Logins and Passwords'' management page</a>. Issues with filling and saving passwords on websites belong in <a href="https://wiki.mozilla.org/Toolkit:Password_Manager" target="_blank" rel="noopener">Toolkit::Password Manager</a>.

Pick the team whose components best match the bug, and answer with the team name only (the text after TEAM:).'
WHERE 'v3' NOT IN (SELECT version FROM support.bugzilla.prompts);

In [0]:
-- ===== Cell 2 (run every time) =====
-- Uses three notebook parameters (they show up as boxes at the top of the notebook):
--   :model           e.g. databricks-gpt-oss-20b
--   :prompt_version  e.g. v2
--   :eval_table      e.g. support.bugzilla.eval_set
INSERT INTO support.bugzilla.eval_runs
SELECT current_timestamp(), :model, :prompt_version, :eval_table,
       id, team, is_urgent, response.result, response.errorMessage
FROM (
  SELECT e.id, e.team, e.is_urgent,
    ai_query(:model,
      CONCAT(p.team_guide,
             '\n\nBug title: ', e.summary,
             '\nBug description: ', substr(e.description, 1, 2000),
             '\n\nWhich team should handle this bug? Answer with the exact team name from the list and nothing else.'),
      modelParameters => named_struct('max_tokens', 2000, 'temperature', 0.0),
      failOnError => false) AS response
  FROM IDENTIFIER(:eval_table) e
  JOIN support.bugzilla.prompts p ON p.version = :prompt_version
);

In [0]:
-- ===== Cell 3 (your scoreboard) =====
-- One query over eval_runs, grouped by model, prompt_version and eval_table,
-- gives accuracy for every run you have ever done. Yours to write.
SELECT
    model, prompt_version, eval_table, run_at,
    SUM(CASE WHEN predicted = true_team THEN 1 ELSE 0 END) / COUNT(*) AS accuracy
FROM support.bugzilla.eval_runs
GROUP BY model, prompt_version, eval_table, run_at
ORDER BY model, run_at

In [0]:
SELECT
  true_team,
  COUNT(*) AS bugs,
  ROUND(AVG(CASE WHEN predicted = true_team THEN 1 ELSE 0 END) * 100, 1) AS accuracy_pct
FROM support.bugzilla.eval_runs
WHERE model = :model
  AND prompt_version = :prompt_version
  AND eval_table = :eval_table
  AND run_at = (
    SELECT MAX(run_at) FROM support.bugzilla.eval_runs
    WHERE model = :model
      AND prompt_version = :prompt_version
      AND eval_table = :eval_table
  )
GROUP BY true_team
ORDER BY accuracy_pct DESC

In [0]:
SELECT * FROM support.bugzilla.eval_runs